# Interactive Power Systems Pedagogy: Single-Phase and Three-Phase Analysis

This notebook provides interactive visualizations for understanding power flow in AC systems.

## Overview

We'll explore:
1. **Single-Phase Power** - voltage, current, complex power, and time-domain behavior
2. **Three-Phase Power** - balanced systems with sequence analysis

All calculations use RMS (Root Mean Square) values for phasors unless otherwise noted.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from ipywidgets import interact, FloatSlider, IntSlider, Layout
import ipywidgets as widgets

# Set up matplotlib for better looking plots
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## Single-Phase Power Analysis

### Fundamental Equations

#### Phasor Representation (RMS values)

$$\bar{V} = |V| e^{j\theta_V} = |V| \angle \theta_V$$

$$\bar{I} = |I| e^{j\theta_I} = |I| \angle \theta_I$$

#### Time Domain Signals (instantaneous values)

$$v(t) = \sqrt{2} |V| \cos(\omega t + \theta_V)$$

$$i(t) = \sqrt{2} |I| \cos(\omega t + \theta_I)$$

Note: The $\sqrt{2}$ factor converts RMS to peak amplitude.

#### Complex Power

$$\bar{S} = \bar{V} \bar{I}^* = |V||I| e^{j(\theta_V - \theta_I)}$$

$$\bar{S} = P + jQ$$

where:
- $P = |V||I|\cos(\theta_V - \theta_I)$ is the **real power** (watts)
- $Q = |V||I|\sin(\theta_V - \theta_I)$ is the **reactive power** (vars)
- $|S| = |V||I|$ is the **apparent power** (VA)

#### Power Factor

$$\text{PF} = \cos(\theta_V - \theta_I) = \frac{P}{|S|}$$

Leading PF: Current leads voltage ($\theta_I > \theta_V$, capacitive)

Lagging PF: Current lags voltage ($\theta_I < \theta_V$, inductive)

#### Instantaneous Power

$$p(t) = v(t) \cdot i(t) = 2|V||I|\cos(\omega t + \theta_V)\cos(\omega t + \theta_I)$$

Using the product-to-sum identity:

$$p(t) = |V||I|\cos(\theta_V - \theta_I) + |V||I|\cos(2\omega t + \theta_V + \theta_I)$$

$$p(t) = P + |V||I|\cos(2\omega t + \theta_V + \theta_I)$$

The average power is the DC component: $P_{avg} = P$

In [ ]:
def single_phase_analysis(V_mag=120, V_angle=0, I_mag=10, I_angle=-30, 
                         freq=60, integration_cycles=1, time_cycles=3):
    """
    Interactive single-phase power analysis
    
    Parameters:
    - V_mag: Voltage magnitude (RMS) in volts
    - V_angle: Voltage angle in degrees
    - I_mag: Current magnitude (RMS) in amperes
    - I_angle: Current angle in degrees
    - freq: Frequency in Hz
    - integration_cycles: Number of cycles for averaging window
    - time_cycles: Number of cycles to display in time domain
    """
    
    # Convert angles to radians
    theta_V = np.deg2rad(V_angle)
    theta_I = np.deg2rad(I_angle)
    
    # Phasor representation
    V = V_mag * np.exp(1j * theta_V)
    I = I_mag * np.exp(1j * theta_I)
    
    # Complex power
    S = V * np.conj(I)
    P = S.real
    Q = S.imag
    S_mag = np.abs(S)
    
    # Power factor
    PF = np.cos(theta_V - theta_I)
    PF_type = "Leading" if (theta_I > theta_V) else "Lagging" if (theta_I < theta_V) else "Unity"
    
    # Time domain
    omega = 2 * np.pi * freq
    T = 1 / freq
    t = np.linspace(0, time_cycles * T, 1000)
    
    # Instantaneous values (peak = sqrt(2) * RMS)
    v_t = np.sqrt(2) * V_mag * np.cos(omega * t + theta_V)
    i_t = np.sqrt(2) * I_mag * np.cos(omega * t + theta_I)
    p_t = v_t * i_t
    
    # Average power over integration window
    integration_window = integration_cycles * T
    p_avg = np.zeros_like(t)
    for idx in range(len(t)):
        if t[idx] >= integration_window:
            mask = (t >= t[idx] - integration_window) & (t <= t[idx])
            p_avg[idx] = np.mean(p_t[mask])
        else:
            mask = t <= t[idx]
            p_avg[idx] = np.mean(p_t[mask])
    
    # Create figure with subplots
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # 1. Phasor diagram (V and I)
    ax1 = fig.add_subplot(gs[0, 0])
    max_mag = max(V_mag, I_mag) * 1.3
    
    # Draw phasors
    ax1.arrow(0, 0, V.real, V.imag, head_width=max_mag*0.05, head_length=max_mag*0.08,
              fc='blue', ec='blue', linewidth=2, label=f'V = {V_mag}∠{V_angle}° V')
    ax1.arrow(0, 0, I.real*10, I.imag*10, head_width=max_mag*0.05, head_length=max_mag*0.08,
              fc='red', ec='red', linewidth=2, label=f'I = {I_mag}∠{I_angle}° A (×10)')
    
    ax1.set_xlim(-max_mag, max_mag)
    ax1.set_ylim(-max_mag, max_mag)
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=0, color='k', linewidth=0.5)
    ax1.axvline(x=0, color='k', linewidth=0.5)
    ax1.set_xlabel('Real')
    ax1.set_ylabel('Imaginary')
    ax1.set_title('Phasor Diagram: V and I')
    ax1.legend(loc='upper right', fontsize=8)
    
    # 2. Complex Power (S)
    ax2 = fig.add_subplot(gs[0, 1])
    S_max = S_mag * 1.3
    
    ax2.arrow(0, 0, P, Q, head_width=S_max*0.05, head_length=S_max*0.08,
              fc='green', ec='green', linewidth=2)
    
    # Draw P and Q components
    ax2.plot([0, P], [0, 0], 'b--', linewidth=1.5, label=f'P = {P:.1f} W')
    ax2.plot([P, P], [0, Q], 'r--', linewidth=1.5, label=f'Q = {Q:.1f} var')
    
    ax2.set_xlim(-S_max, S_max)
    ax2.set_ylim(-S_max, S_max)
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=0, color='k', linewidth=0.5)
    ax2.axvline(x=0, color='k', linewidth=0.5)
    ax2.set_xlabel('P (Real Power) [W]')
    ax2.set_ylabel('Q (Reactive Power) [var]')
    ax2.set_title(f'Complex Power\n|S| = {S_mag:.1f} VA')
    ax2.legend(loc='upper right', fontsize=8)
    
    # 3. Power Factor Display
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.axis('off')
    
    info_text = f"""Power Factor Analysis
    
PF = {PF:.4f} ({PF_type})
θ = {np.rad2deg(theta_V - theta_I):.2f}°

Real Power (P): {P:.2f} W
Reactive Power (Q): {Q:.2f} var
Apparent Power (|S|): {S_mag:.2f} VA

Load Characteristic:
{"Capacitive (leading)" if Q < 0 else "Inductive (lagging)" if Q > 0 else "Resistive"}
"""
    ax3.text(0.1, 0.5, info_text, transform=ax3.transAxes,
             fontsize=11, verticalalignment='center', family='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # 4. Time domain v(t) and i(t)
    ax4 = fig.add_subplot(gs[1, :])
    ax4.plot(t * 1000, v_t, 'b-', linewidth=2, label='v(t)')
    ax4.plot(t * 1000, i_t * 10, 'r-', linewidth=2, label='i(t) × 10')
    ax4.axhline(y=0, color='k', linewidth=0.5)
    ax4.grid(True, alpha=0.3)
    ax4.set_xlabel('Time [ms]')
    ax4.set_ylabel('Amplitude')
    ax4.set_title('Time Domain: Voltage and Current')
    ax4.legend(loc='upper right')
    
    # 5. Time domain p(t) with average
    ax5 = fig.add_subplot(gs[2, :])
    ax5.plot(t * 1000, p_t, 'g-', linewidth=2, label='p(t) = v(t) · i(t)', alpha=0.7)
    ax5.plot(t * 1000, p_avg, 'k--', linewidth=2.5, 
             label=f'Average over {integration_cycles} cycle(s) = {P:.2f} W')
    ax5.axhline(y=P, color='orange', linewidth=1, linestyle=':', 
                label=f'Theoretical P = {P:.2f} W')
    ax5.axhline(y=0, color='k', linewidth=0.5)
    ax5.fill_between(t * 1000, 0, p_t, where=(p_t >= 0), alpha=0.2, color='green', 
                     label='Positive power')
    ax5.fill_between(t * 1000, 0, p_t, where=(p_t < 0), alpha=0.2, color='red',
                     label='Negative power')
    ax5.grid(True, alpha=0.3)
    ax5.set_xlabel('Time [ms]')
    ax5.set_ylabel('Power [W]')
    ax5.set_title('Instantaneous Power p(t) = v(t) · i(t)')
    ax5.legend(loc='upper right')
    
    plt.suptitle('Single-Phase Power Analysis', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Create interactive widget
slider_layout = Layout(width='500px')

interact(single_phase_analysis,
         V_mag=FloatSlider(min=1, max=480, step=1, value=120, description='|V| (RMS)', layout=slider_layout),
         V_angle=FloatSlider(min=-180, max=180, step=5, value=0, description='∠V (deg)', layout=slider_layout),
         I_mag=FloatSlider(min=0.1, max=50, step=0.1, value=10, description='|I| (RMS)', layout=slider_layout),
         I_angle=FloatSlider(min=-180, max=180, step=5, value=-30, description='∠I (deg)', layout=slider_layout),
         freq=FloatSlider(min=50, max=400, step=10, value=60, description='Freq (Hz)', layout=slider_layout),
         integration_cycles=IntSlider(min=1, max=10, step=1, value=1, 
                                     description='Avg Window', layout=slider_layout),
         time_cycles=IntSlider(min=1, max=10, step=1, value=3, 
                              description='Time Span', layout=slider_layout));

## Three-Phase Power Analysis

### Balanced Three-Phase Systems

#### Phase Voltages (Y-connected, Line-to-Neutral)

$$\bar{V}_a = |V_{LN}| \angle \theta_V$$

$$\bar{V}_b = |V_{LN}| \angle (\theta_V - 120°)$$

$$\bar{V}_c = |V_{LN}| \angle (\theta_V + 120°)$$

Line-to-Line voltage magnitude: $|V_{LL}| = \sqrt{3} |V_{LN}|$

#### Phase Currents (Balanced Load)

$$\bar{I}_a = |I| \angle \theta_I$$

$$\bar{I}_b = |I| \angle (\theta_I - 120°)$$

$$\bar{I}_c = |I| \angle (\theta_I + 120°)$$

#### Three-Phase Complex Power

$$\bar{S}_{3\phi} = \bar{V}_a \bar{I}_a^* + \bar{V}_b \bar{I}_b^* + \bar{V}_c \bar{I}_c^*$$

For balanced systems:

$$\bar{S}_{3\phi} = 3 \bar{V}_a \bar{I}_a^* = 3 |V_{LN}||I| e^{j(\theta_V - \theta_I)}$$

$$P_{3\phi} = 3 |V_{LN}||I| \cos(\theta_V - \theta_I) = \sqrt{3} |V_{LL}||I| \cos(\theta_V - \theta_I)$$

$$Q_{3\phi} = 3 |V_{LN}||I| \sin(\theta_V - \theta_I) = \sqrt{3} |V_{LL}||I| \sin(\theta_V - \theta_I)$$

#### Instantaneous Three-Phase Power

$$p_{3\phi}(t) = p_a(t) + p_b(t) + p_c(t)$$

For balanced systems, instantaneous power is **constant**:

$$p_{3\phi}(t) = P_{3\phi} = \text{constant}$$

This is a key advantage of three-phase systems!

#### Symmetrical Components

Any unbalanced set of phasors can be decomposed into:
- **Positive sequence**: rotates in abc direction
- **Negative sequence**: rotates in acb direction  
- **Zero sequence**: all phases in phase

Operator: $\mathbf{a} = e^{j120°} = -0.5 + j0.866$

$$\begin{bmatrix} \bar{V}_0 \\ \bar{V}_+ \\ \bar{V}_- \end{bmatrix} = \frac{1}{3} \begin{bmatrix} 1 & 1 & 1 \\ 1 & \mathbf{a} & \mathbf{a}^2 \\ 1 & \mathbf{a}^2 & \mathbf{a} \end{bmatrix} \begin{bmatrix} \bar{V}_a \\ \bar{V}_b \\ \bar{V}_c \end{bmatrix}$$

In [ ]:
def three_phase_analysis(V_LN_mag=120, V_angle=0, I_mag=10, I_angle=-30,
                        freq=60, integration_cycles=1, time_cycles=3,
                        show_line_voltage=True):
    """
    Interactive three-phase power analysis (balanced system)
    
    Parameters:
    - V_LN_mag: Line-to-neutral voltage magnitude (RMS)
    - V_angle: Voltage angle for phase A
    - I_mag: Current magnitude (RMS) per phase
    - I_angle: Current angle for phase A
    - freq: Frequency in Hz
    - integration_cycles: Number of cycles for averaging
    - time_cycles: Number of cycles to display
    - show_line_voltage: Show line-to-line voltages on phasor diagram
    """
    
    # Convert to radians
    theta_V = np.deg2rad(V_angle)
    theta_I = np.deg2rad(I_angle)
    
    # Phase voltages (line-to-neutral)
    V_a = V_LN_mag * np.exp(1j * theta_V)
    V_b = V_LN_mag * np.exp(1j * (theta_V - 2*np.pi/3))
    V_c = V_LN_mag * np.exp(1j * (theta_V + 2*np.pi/3))
    
    # Line-to-line voltages
    V_ab = V_a - V_b
    V_bc = V_b - V_c
    V_ca = V_c - V_a
    V_LL_mag = np.abs(V_ab)
    
    # Phase currents
    I_a = I_mag * np.exp(1j * theta_I)
    I_b = I_mag * np.exp(1j * (theta_I - 2*np.pi/3))
    I_c = I_mag * np.exp(1j * (theta_I + 2*np.pi/3))
    
    # Complex power (per phase and total)
    S_a = V_a * np.conj(I_a)
    S_b = V_b * np.conj(I_b)
    S_c = V_c * np.conj(I_c)
    S_3ph = S_a + S_b + S_c
    
    P_3ph = S_3ph.real
    Q_3ph = S_3ph.imag
    S_3ph_mag = np.abs(S_3ph)
    
    # Power factor
    PF = np.cos(theta_V - theta_I)
    PF_type = "Leading" if (theta_I > theta_V) else "Lagging" if (theta_I < theta_V) else "Unity"
    
    # Time domain
    omega = 2 * np.pi * freq
    T = 1 / freq
    t = np.linspace(0, time_cycles * T, 2000)
    
    # Instantaneous voltages and currents
    v_a_t = np.sqrt(2) * V_LN_mag * np.cos(omega * t + theta_V)
    v_b_t = np.sqrt(2) * V_LN_mag * np.cos(omega * t + theta_V - 2*np.pi/3)
    v_c_t = np.sqrt(2) * V_LN_mag * np.cos(omega * t + theta_V + 2*np.pi/3)
    
    i_a_t = np.sqrt(2) * I_mag * np.cos(omega * t + theta_I)
    i_b_t = np.sqrt(2) * I_mag * np.cos(omega * t + theta_I - 2*np.pi/3)
    i_c_t = np.sqrt(2) * I_mag * np.cos(omega * t + theta_I + 2*np.pi/3)
    
    # Instantaneous power per phase
    p_a_t = v_a_t * i_a_t
    p_b_t = v_b_t * i_b_t
    p_c_t = v_c_t * i_c_t
    p_3ph_t = p_a_t + p_b_t + p_c_t
    
    # Average power
    integration_window = integration_cycles * T
    p_avg = np.zeros_like(t)
    for idx in range(len(t)):
        if t[idx] >= integration_window:
            mask = (t >= t[idx] - integration_window) & (t <= t[idx])
            p_avg[idx] = np.mean(p_3ph_t[mask])
        else:
            mask = t <= t[idx]
            p_avg[idx] = np.mean(p_3ph_t[mask])
    
    # Create figure
    fig = plt.figure(figsize=(18, 14))
    gs = fig.add_gridspec(4, 3, hspace=0.35, wspace=0.3)
    
    # 1. Voltage phasor diagram
    ax1 = fig.add_subplot(gs[0, 0])
    max_V = V_LN_mag * 2.2
    
    # Phase voltages
    ax1.arrow(0, 0, V_a.real, V_a.imag, head_width=max_V*0.04, head_length=max_V*0.06,
              fc='blue', ec='blue', linewidth=2, label=f'Va = {V_LN_mag}∠{V_angle}°')
    ax1.arrow(0, 0, V_b.real, V_b.imag, head_width=max_V*0.04, head_length=max_V*0.06,
              fc='green', ec='green', linewidth=2, label=f'Vb')
    ax1.arrow(0, 0, V_c.real, V_c.imag, head_width=max_V*0.04, head_length=max_V*0.06,
              fc='red', ec='red', linewidth=2, label=f'Vc')
    
    # Line-to-line voltages (optional)
    if show_line_voltage:
        ax1.arrow(0, 0, V_ab.real, V_ab.imag, head_width=max_V*0.04, head_length=max_V*0.06,
                  fc='cyan', ec='cyan', linewidth=1.5, linestyle='--', alpha=0.6, label=f'Vab')
    
    ax1.set_xlim(-max_V, max_V)
    ax1.set_ylim(-max_V, max_V)
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)
    ax1.axhline(y=0, color='k', linewidth=0.5)
    ax1.axvline(x=0, color='k', linewidth=0.5)
    ax1.set_xlabel('Real')
    ax1.set_ylabel('Imaginary')
    ax1.set_title(f'Voltage Phasors\nVLN = {V_LN_mag:.1f}V, VLL = {V_LL_mag:.1f}V')
    ax1.legend(loc='upper right', fontsize=8)
    
    # 2. Current phasor diagram
    ax2 = fig.add_subplot(gs[0, 1])
    max_I = I_mag * 1.5
    
    ax2.arrow(0, 0, I_a.real, I_a.imag, head_width=max_I*0.04, head_length=max_I*0.06,
              fc='blue', ec='blue', linewidth=2, label=f'Ia = {I_mag}∠{I_angle}°')
    ax2.arrow(0, 0, I_b.real, I_b.imag, head_width=max_I*0.04, head_length=max_I*0.06,
              fc='green', ec='green', linewidth=2, label=f'Ib')
    ax2.arrow(0, 0, I_c.real, I_c.imag, head_width=max_I*0.04, head_length=max_I*0.06,
              fc='red', ec='red', linewidth=2, label=f'Ic')
    
    ax2.set_xlim(-max_I, max_I)
    ax2.set_ylim(-max_I, max_I)
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=0, color='k', linewidth=0.5)
    ax2.axvline(x=0, color='k', linewidth=0.5)
    ax2.set_xlabel('Real')
    ax2.set_ylabel('Imaginary')
    ax2.set_title('Current Phasors (Balanced)')
    ax2.legend(loc='upper right', fontsize=8)
    
    # 3. Three-phase complex power
    ax3 = fig.add_subplot(gs[0, 2])
    S_max = S_3ph_mag * 1.3
    
    ax3.arrow(0, 0, P_3ph, Q_3ph, head_width=S_max*0.04, head_length=S_max*0.06,
              fc='purple', ec='purple', linewidth=2.5)
    ax3.plot([0, P_3ph], [0, 0], 'b--', linewidth=1.5, label=f'P = {P_3ph:.1f} W')
    ax3.plot([P_3ph, P_3ph], [0, Q_3ph], 'r--', linewidth=1.5, label=f'Q = {Q_3ph:.1f} var')
    
    ax3.set_xlim(-S_max, S_max)
    ax3.set_ylim(-S_max, S_max)
    ax3.set_aspect('equal')
    ax3.grid(True, alpha=0.3)
    ax3.axhline(y=0, color='k', linewidth=0.5)
    ax3.axvline(x=0, color='k', linewidth=0.5)
    ax3.set_xlabel('P (Real Power) [W]')
    ax3.set_ylabel('Q (Reactive Power) [var]')
    ax3.set_title(f'3φ Complex Power\n|S| = {S_3ph_mag:.1f} VA')
    ax3.legend(loc='upper right', fontsize=8)
    
    # 4. Power summary
    ax4 = fig.add_subplot(gs[1, 0])
    ax4.axis('off')
    
    info_text = f"""Three-Phase Power Summary
    
Power Factor: {PF:.4f} ({PF_type})
θ = {np.rad2deg(theta_V - theta_I):.2f}°

Total 3φ Power:
  P = {P_3ph:.2f} W
  Q = {Q_3ph:.2f} var
  |S| = {S_3ph_mag:.2f} VA

Per Phase:
  P = {P_3ph/3:.2f} W
  Q = {Q_3ph/3:.2f} var
  |S| = {S_3ph_mag/3:.2f} VA
"""
    ax4.text(0.1, 0.5, info_text, transform=ax4.transAxes,
             fontsize=10, verticalalignment='center', family='monospace',
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    
    # 5. Symmetrical components
    ax5 = fig.add_subplot(gs[1, 1])
    ax5.axis('off')
    
    # Calculate sequence components
    a = np.exp(1j * 2 * np.pi / 3)  # 120° operator
    V_0 = (V_a + V_b + V_c) / 3
    V_pos = (V_a + a * V_b + a**2 * V_c) / 3
    V_neg = (V_a + a**2 * V_b + a * V_c) / 3
    
    I_0 = (I_a + I_b + I_c) / 3
    I_pos = (I_a + a * I_b + a**2 * I_c) / 3
    I_neg = (I_a + a**2 * I_b + a * I_c) / 3
    
    seq_text = f"""Symmetrical Components

Voltage Sequences:
  V₀ = {np.abs(V_0):.3f}∠{np.angle(V_0, deg=True):.1f}°
  V₊ = {np.abs(V_pos):.3f}∠{np.angle(V_pos, deg=True):.1f}°
  V₋ = {np.abs(V_neg):.3f}∠{np.angle(V_neg, deg=True):.1f}°

Current Sequences:
  I₀ = {np.abs(I_0):.3f}∠{np.angle(I_0, deg=True):.1f}°
  I₊ = {np.abs(I_pos):.3f}∠{np.angle(I_pos, deg=True):.1f}°
  I₋ = {np.abs(I_neg):.3f}∠{np.angle(I_neg, deg=True):.1f}°

Balanced ⟹ V₀≈0, V₋≈0, I₀≈0, I₋≈0
"""
    ax5.text(0.1, 0.5, seq_text, transform=ax5.transAxes,
             fontsize=9, verticalalignment='center', family='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))
    
    # 6. Time domain voltages
    ax6 = fig.add_subplot(gs[2, :])
    ax6.plot(t * 1000, v_a_t, 'b-', linewidth=1.5, label='va(t)', alpha=0.8)
    ax6.plot(t * 1000, v_b_t, 'g-', linewidth=1.5, label='vb(t)', alpha=0.8)
    ax6.plot(t * 1000, v_c_t, 'r-', linewidth=1.5, label='vc(t)', alpha=0.8)
    ax6.axhline(y=0, color='k', linewidth=0.5)
    ax6.grid(True, alpha=0.3)
    ax6.set_xlabel('Time [ms]')
    ax6.set_ylabel('Voltage [V]')
    ax6.set_title('Time Domain: Three-Phase Voltages')
    ax6.legend(loc='upper right')
    
    # 7. Time domain currents
    ax7 = fig.add_subplot(gs[3, 0])
    ax7.plot(t * 1000, i_a_t, 'b-', linewidth=1.5, label='ia(t)', alpha=0.8)
    ax7.plot(t * 1000, i_b_t, 'g-', linewidth=1.5, label='ib(t)', alpha=0.8)
    ax7.plot(t * 1000, i_c_t, 'r-', linewidth=1.5, label='ic(t)', alpha=0.8)
    ax7.axhline(y=0, color='k', linewidth=0.5)
    ax7.grid(True, alpha=0.3)
    ax7.set_xlabel('Time [ms]')
    ax7.set_ylabel('Current [A]')
    ax7.set_title('Three-Phase Currents')
    ax7.legend(loc='upper right')
    
    # 8. Per-phase power
    ax8 = fig.add_subplot(gs[3, 1])
    ax8.plot(t * 1000, p_a_t, 'b-', linewidth=1, label='pa(t)', alpha=0.6)
    ax8.plot(t * 1000, p_b_t, 'g-', linewidth=1, label='pb(t)', alpha=0.6)
    ax8.plot(t * 1000, p_c_t, 'r-', linewidth=1, label='pc(t)', alpha=0.6)
    ax8.axhline(y=0, color='k', linewidth=0.5)
    ax8.grid(True, alpha=0.3)
    ax8.set_xlabel('Time [ms]')
    ax8.set_ylabel('Power [W]')
    ax8.set_title('Per-Phase Instantaneous Power')
    ax8.legend(loc='upper right')
    
    # 9. Total three-phase power (constant!)
    ax9 = fig.add_subplot(gs[3, 2])
    ax9.plot(t * 1000, p_3ph_t, 'purple', linewidth=2.5, label='p3φ(t) = pa + pb + pc')
    ax9.plot(t * 1000, p_avg, 'k--', linewidth=2, 
             label=f'Average = {P_3ph:.2f} W')
    ax9.axhline(y=P_3ph, color='orange', linewidth=1, linestyle=':', 
                label=f'Theoretical P = {P_3ph:.2f} W')
    ax9.axhline(y=0, color='k', linewidth=0.5)
    ax9.grid(True, alpha=0.3)
    ax9.set_xlabel('Time [ms]')
    ax9.set_ylabel('Power [W]')
    ax9.set_title('Total 3φ Instantaneous Power (Constant for Balanced Loads!)')
    ax9.legend(loc='upper right', fontsize=8)
    
    plt.suptitle('Three-Phase Power Analysis (Balanced System)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Create interactive widget
interact(three_phase_analysis,
         V_LN_mag=FloatSlider(min=1, max=480, step=1, value=120, description='|VLN| (RMS)', layout=slider_layout),
         V_angle=FloatSlider(min=-180, max=180, step=5, value=0, description='∠Va (deg)', layout=slider_layout),
         I_mag=FloatSlider(min=0.1, max=50, step=0.1, value=10, description='|I| (RMS)', layout=slider_layout),
         I_angle=FloatSlider(min=-180, max=180, step=5, value=-30, description='∠Ia (deg)', layout=slider_layout),
         freq=FloatSlider(min=50, max=400, step=10, value=60, description='Freq (Hz)', layout=slider_layout),
         integration_cycles=IntSlider(min=1, max=10, step=1, value=1, 
                                     description='Avg Window', layout=slider_layout),
         time_cycles=IntSlider(min=1, max=10, step=1, value=3, 
                              description='Time Span', layout=slider_layout),
         show_line_voltage=widgets.Checkbox(value=False, description='Show VLL'));

## Key Observations and Learning Points

### Single-Phase Systems

1. **Instantaneous power pulsates** at twice the line frequency (2ω)
2. Power factor affects both the average power and the amplitude of pulsation
3. Reactive power (Q) represents energy oscillating between source and load
4. Pure resistive loads (PF = 1) have zero reactive power
5. The time-average of p(t) equals the real power P

### Three-Phase Systems

1. **Constant instantaneous power** - the pulsations in each phase cancel out!
2. This results in smooth mechanical torque in motors and less vibration
3. For balanced systems: $P_{3\phi} = 3 P_{1\phi}$ and $Q_{3\phi} = 3 Q_{1\phi}$
4. Line-to-line voltage is √3 times line-to-neutral voltage
5. Symmetrical components reveal that balanced systems have only positive sequence
6. Zero sequence currents require a neutral return path

### Power Factor Insights

- **Leading PF** (capacitive): Current leads voltage, Q < 0
- **Lagging PF** (inductive): Current lags voltage, Q > 0  
- **Unity PF** (resistive): Current in phase with voltage, Q = 0

Low power factor means:
- Higher current for same real power
- Increased I²R losses in transmission
- Larger equipment ratings needed

## Exercises

Try these experiments with the sliders:

1. Set PF = 1 (make ∠I = ∠V) and observe zero reactive power
2. Create a purely reactive load (∠I = ∠V ± 90°) and watch P approach zero
3. Compare single-phase p(t) ripple with constant three-phase p3φ(t)
4. Adjust the integration window to see averaging behavior
5. Change frequency to see how cycle time affects the time-domain plots

## References

- Grainger & Stevenson, *Power System Analysis*
- Krause, Wasynczuk & Sudhoff, *Analysis of Electric Machinery*
- IEEE Standard 1459-2010: Definitions for Measurement of Electric Power